# Network Meta-Analysis

**Import datasets after extraction and transformation**.

In [3]:
import pandas as pd
csv = ["continuous"]
for x in csv:
    df = pd.read_csv(f"./data/{x}.csv", encoding = "utf-8")
    globals()[x] = df

### Continuous

In [4]:
continuous = continuous.fillna("")
continuous.set_index(keys = ["id"], inplace=True)

In [5]:
df = continuous 
df = df.drop(columns = ["min", "q1", "median", "q3", "max"])

ikdc_subjective = df[df["outcome"] == "ikdc"]
ikdc_subjective["outcome"] = "ikdc_subjective"
ikdc_subjective.to_csv("./data/ikdc_subjective.csv", encoding = "utf-8")

lysholm = df[df["outcome"] == "lysholm"]
lysholm.to_csv("./data/lysholm.csv", encoding = "utf-8")

tegner = df[df["outcome"] == "tegner"]
tegner.to_csv("./data/tegner.csv", encoding = "utf-8")

instrumented_laxity = df[df["outcome"] == "instrumented_laxity"]
instrumented_laxity.to_csv("./data/instrumented_laxity.csv", encoding = "utf-8")

In [6]:
import pandas as pd

continuous = [
    "ikdc_subjective",
    "lysholm",
    "tegner",
    "instrumented_laxity"
]

dichotomous = [
    "pivot_shift",
    "lachman",
    "graft_failure"
]

for x in continuous:
    df = pd.read_csv(f"./data/{x}.csv", encoding = "utf-8")
    globals()[x] = df

In [7]:
ikdc_subjective.head()
len(ikdc_subjective)

52

In [62]:
dfs = {
    "ikdc_subjective": ikdc_subjective, 
    "lysholm": lysholm, 
    "tegner": tegner, 
    "instrumented_laxity": instrumented_laxity
}

for name, df in dfs.items():
    df["group"] = df.groupby(["study"]).cumcount() + 1
    
    # reshape
    wide = (
        df.pivot(
            index= "study",
            columns="group",
            values=["subgroup", "n", "mean", "sd"]
        )
    )
    
    # flatten column names
    wide.columns = [
        f"{name}{group}" if name != "subgroup" else f"treat{group}"
        for name, group in wide.columns
    ]
    
    # rename variables
    wide = (
        wide.rename(columns={
            "n1": "n1",
            "mean1": "m1",
            "sd1": "sd1",
            "n2": "n2",
            "mean2": "m2",
            "sd2": "sd2",
        })
        .reset_index()
    )

    wide["n1"] = wide["n1"].astype(int)
    wide["n2"] = wide["n2"].astype(int)
    wide["sd1"] = wide["sd1"].astype(float)
    wide["sd2"] =wide["sd2"].astype(float)
    wide["m1"] = wide["m1"].astype(float)
    wide["m2"] = wide["m2"].astype(float)
    
    import os
    os.makedirs(f"./analysis", exist_ok = True)
    wide.to_csv(f"./analysis/{name}.csv", encoding = "utf-8")
    globals()[name] = wide

In [64]:
ikdc_subjective.head()

,study,treat1,treat2,n1,n2,m1,m2,sd1,sd2
0,"Barié et al., 2020",BPTB,QT,43,43,91.0,92.0,7.3,11.5
1,"Bi et al., 2018",HT,PLT,62,62,90.4,89.3,7.1,8.4
2,"Björnsson et al., 2016",BPTB,HT,61,86,67.3,74.0,20.8,18.8
3,"Bottoni et al., 2015",HT,TA,44,36,79.5,81.1,24.7,23.1
4,"Butt et al., 2024",HT,PLT,30,30,89.7,89.9,5.7,9.8


In [65]:
import numpy as np

# Standardized Mean Difference (SMD)
def SMD(n1, n2, m1, m2, sd1, sd2):
    sd = (
        (((n1 - 1) * sd1**2) + ((n2 - 1) * sd2**2))
        / (n1 + n2 - 2)
    )
    sp = np.sqrt(sd)
    return round((np.abs(m1 - m2) / sp), 3)

# Standard Error of the Standardized Mean Difference (SMD)
def SE(n1, n2, smd):
    se = np.sqrt(
        (n1 + n2)/(n1*n2)
        + smd**2/(2*(n1+n2-2))
    )
    se = round(se, 3)
    return se

#### IKDC subjective

In [73]:
df = ikdc_subjective
df["smd"] = SMD(df["n1"], df["n2"],df["m1"], df["m2"],df["sd1"], df["sd2"])
df["se"] = SE(df["n1"], df["n2"], df["smd"])

df.to_csv(f"./analysis/ikdc_subjective.csv")
#df.to_excel(f"./analysis/ikdc_subjective.xlsx")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 26 entries, 0 to 25
Data columns (total 11 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   study   26 non-null     str    
 1   treat1  26 non-null     str    
 2   treat2  26 non-null     str    
 3   n1      26 non-null     int64  
 4   n2      26 non-null     int64  
 5   m1      26 non-null     float64
 6   m2      26 non-null     float64
 7   sd1     26 non-null     float64
 8   sd2     26 non-null     float64
 9   smd     26 non-null     float64
 10  se      26 non-null     float64
dtypes: float64(6), int64(2), str(3)
memory usage: 3.0 KB


#### Lysholm

In [72]:
df = pd.read_csv("./data/lysholm.csv", encoding = "utf-8")
df = lysholm.fillna("")
df["smd"] = SMD(df["n1"], df["n2"],df["m1"], df["m2"],df["sd1"], df["sd2"])
df["se"] = SE(df["n1"], df["n2"], df["smd"])

df.to_csv(f"./analysis/lysholm.csv")
#df.to_excel(f"./analysis/lysholm.xlsx")

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 33 entries, 0 to 32
Data columns (total 11 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   study   33 non-null     str    
 1   treat1  33 non-null     str    
 2   treat2  33 non-null     str    
 3   n1      33 non-null     int64  
 4   n2      33 non-null     int64  
 5   m1      33 non-null     float64
 6   m2      33 non-null     float64
 7   sd1     33 non-null     float64
 8   sd2     33 non-null     float64
 9   smd     33 non-null     float64
 10  se      33 non-null     float64
dtypes: float64(6), int64(2), str(3)
memory usage: 3.8 KB


#### Tegner

In [71]:
df["smd"] = SMD(df["n1"], df["n2"],df["m1"], df["m2"],df["sd1"], df["sd2"])
df["se"] = SE(df["n1"], df["n2"], df["smd"])

df.to_csv(f"./analysis/tegner.csv")
#df.to_excel(f"./analysis/tegner.xlsx")

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 34 entries, 0 to 33
Data columns (total 11 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   study   34 non-null     str    
 1   treat1  34 non-null     str    
 2   treat2  34 non-null     str    
 3   n1      34 non-null     int64  
 4   n2      34 non-null     int64  
 5   m1      34 non-null     float64
 6   m2      34 non-null     float64
 7   sd1     34 non-null     float64
 8   sd2     34 non-null     float64
 9   smd     34 non-null     float64
 10  se      34 non-null     float64
dtypes: float64(6), int64(2), str(3)
memory usage: 3.9 KB


#### Instrumented laxity

In [70]:
df = instrumented_laxity.fillna("")
df["smd"] = SMD(df["n1"], df["n2"],df["m1"], df["m2"],df["sd1"], df["sd2"])
df["se"] = SE(df["n1"], df["n2"], df["smd"])

df.to_csv(f"./analysis/instrumented_laxity.csv")
#df.to_excel(f"./analysis/instrumented_laxity.xlsx")

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 34 entries, 0 to 33
Data columns (total 11 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   study   34 non-null     str    
 1   treat1  34 non-null     str    
 2   treat2  34 non-null     str    
 3   n1      34 non-null     int64  
 4   n2      34 non-null     int64  
 5   m1      34 non-null     float64
 6   m2      34 non-null     float64
 7   sd1     34 non-null     float64
 8   sd2     34 non-null     float64
 9   smd     34 non-null     float64
 10  se      34 non-null     float64
dtypes: float64(6), int64(2), str(3)
memory usage: 3.9 KB


### DIchotomous

#### Pivot-shift

In [78]:
pivot_shift.head()

,id,study,subgroup,outcome,ni,0,1+,2+,3+
0,2,"Aglietti et al., 2004",BPTB,pivot_shift,60,50.0,10.0,0,0
1,2,"Aglietti et al., 2004",HT,pivot_shift,60,49.0,11.0,0,0
2,5,"Anderson et al., 2001",BPTB,pivot_shift,35,17.0,3.0,8,7
3,5,"Anderson et al., 2001",HT,pivot_shift,70,34.0,9.0,16,11
4,11,"Beynnon et al., 2002",BPTB,pivot_shift,22,19.0,3.0,0,0


In [137]:
df = pivot_shift
df = df.drop(columns = ["outcome"])

In [138]:
df.head()

,id,study,subgroup,ni,0,1+,2+,3+
0,2,"Aglietti et al., 2004",BPTB,60,50.0,10.0,0,0
1,2,"Aglietti et al., 2004",HT,60,49.0,11.0,0,0
2,5,"Anderson et al., 2001",BPTB,35,17.0,3.0,8,7
3,5,"Anderson et al., 2001",HT,70,34.0,9.0,16,11
4,11,"Beynnon et al., 2002",BPTB,22,19.0,3.0,0,0


In [ ]:
df.to_csv("./data/pivot_shift.csv", encoding = "utf-8")

In [139]:
df_1 = df.copy()
df_2 = df.copy()

df_1["xi"] = df_1["1+"] + df_1["2+"] + df_1["3+"]
df_2["xi"] = df_2["2+"] + df_2["3+"]

In [140]:
df_1["yi"] = round(((df_1["xi"] + 0.5) / (df_1["ni"] + 0.5)),3)
df_2["yi"] = round(((df_2["xi"] + 0.5) / (df_2["ni"] + 0.5)),3)

In [147]:
def SE(p, n):
    import numpy as np
    se = (p * (1 - p)) / np.sqrt(n)
    se = round(se, 3)
    return se

def SD(se, n):
    import numpy as np
    sd = se * np.sqrt(n)
    sd = round(sd, 3)
    return sd

In [141]:
p = df_1["yi"]
n = df_1["ni"]

se = SE(p, n)
df_1["se"] = se

sd = SD(se, n)
df_1["sd"] = sd

In [142]:
p = df_2["yi"]
n = df_2["ni"]

se = SE(p, n)
df_2["se"] = se

sd = SD(se, n)
df_2["sd"] = sd

In [159]:
df_1 = df_1.dropna()
df_1.to_csv("./analysis/pivot_shift_1.csv", encoding = "utf-8")
df_2.to_csv("./analysis/pivot_shift_2.csv", encoding = "utf-8")

#### Lachman

In [144]:
lachman.head()

,id,study,subgroup,ni,0,1+,2+,3+,y,xi,yi
0,11,"Beynnon et al., 2002",BPTB,22,10.0,10.0,2,0,2,2,0.111
1,11,"Beynnon et al., 2002",HT,22,3.0,6.0,8,5,13,13,0.600
2,13,"Björnsson et al., 2016",BPTB,61,30.0,24.0,6,1,7,7,0.122
3,13,"Björnsson et al., 2016",HT,86,19.0,46.0,12,9,21,21,0.249
4,23,"Ejerhed et al., 2003",BPTB,32,18.0,12.0,2,0,2,2,0.077


In [145]:
df = lachman
df_1 = lachman.copy()
df_2 = lachman.copy()

df_1["xi"] = df_1["1+"] + df_1["2+"] + df_1["3+"]
df_2["xi"] = df_2["2+"] + df_2["3+"]

In [146]:
df_1["yi"] = round(((df_1["xi"] + 0.5) / (df_1["ni"] + 0.5)),3)
df_2["yi"] = round(((df_2["xi"] + 0.5) / (df_2["ni"] + 0.5)),3)

In [148]:
p = df_1["yi"]
n = df_1["ni"]

se = SE(p, n)
df_1["se"] = se

sd = SD(se, n)
df_1["sd"] = sd

In [149]:
p = df_2["yi"]
n = df_2["ni"]

se = SE(p, n)
df_2["se"] = se

sd = SD(se, n)
df_2["sd"] = sd

In [134]:
df_1.head()

,id,study,subgroup,ni,0,1+,2+,3+,y,xi,yi,se,sd
0,11,"Beynnon et al., 2002",BPTB,22,10.0,10.0,2,0,2,12.0,0.556,0.053,0.249
1,11,"Beynnon et al., 2002",HT,22,3.0,6.0,8,5,13,19.0,0.867,0.025,0.117
2,13,"Björnsson et al., 2016",BPTB,61,30.0,24.0,6,1,7,31.0,0.512,0.032,0.250
3,13,"Björnsson et al., 2016",HT,86,19.0,46.0,12,9,21,67.0,0.780,0.019,0.176
4,23,"Ejerhed et al., 2003",BPTB,32,18.0,12.0,2,0,2,14.0,0.446,0.044,0.249


In [135]:
df_2.head()

,id,study,subgroup,ni,0,1+,2+,3+,y,xi,yi,se,sd
0,11,"Beynnon et al., 2002",BPTB,22,10.0,10.0,2,0,2,2,0.111,0.021,0.098
1,11,"Beynnon et al., 2002",HT,22,3.0,6.0,8,5,13,13,0.600,0.051,0.239
2,13,"Björnsson et al., 2016",BPTB,61,30.0,24.0,6,1,7,7,0.122,0.014,0.109
3,13,"Björnsson et al., 2016",HT,86,19.0,46.0,12,9,21,21,0.249,0.020,0.185
4,23,"Ejerhed et al., 2003",BPTB,32,18.0,12.0,2,0,2,2,0.077,0.013,0.074


,id,study,subgroup,ni,0,1+,2+,3+,y,xi,yi
0,11,"Beynnon et al., 2002",BPTB,22,10.0,10.0,2,0,2,12.0,0.556
1,11,"Beynnon et al., 2002",HT,22,3.0,6.0,8,5,13,19.0,0.867
2,13,"Björnsson et al., 2016",BPTB,61,30.0,24.0,6,1,7,31.0,0.512
3,13,"Björnsson et al., 2016",HT,86,19.0,46.0,12,9,21,67.0,0.780
4,23,"Ejerhed et al., 2003",BPTB,32,18.0,12.0,2,0,2,14.0,0.446
5,23,"Ejerhed et al., 2003",HT,33,16.0,17.0,0,0,0,17.0,0.522
6,26,"Feller & Webster, 2003",BPTB,42,36.0,6.0,0,0,0,6.0,0.153
7,26,"Feller & Webster, 2003",HT,47,28.0,19.0,0,0,0,19.0,0.411
8,30,"Gifstad et al., 2013",BPTB,45,29.0,15.0,1,0,1,16.0,0.363
9,30,"Gifstad et al., 2013",HT,32,19.0,12.0,1,0,1,13.0,0.415


In [106]:
df_2.head()

,id,study,subgroup,ni,0,1+,2+,3+,y,xi,yi
0,11,"Beynnon et al., 2002",BPTB,22,10.0,10.0,2,0,2,2,0.111
1,11,"Beynnon et al., 2002",HT,22,3.0,6.0,8,5,13,13,0.600
2,13,"Björnsson et al., 2016",BPTB,61,30.0,24.0,6,1,7,7,0.122
3,13,"Björnsson et al., 2016",HT,86,19.0,46.0,12,9,21,21,0.249
4,23,"Ejerhed et al., 2003",BPTB,32,18.0,12.0,2,0,2,2,0.077


In [157]:
df_1.to_csv("./analysis/lachman_1.csv", encoding = "utf-8")
df_2.to_csv("./analysis/lachman_2.csv", encoding = "utf-8")

In [160]:
dfs = ["pivot_shift_1", "pivot_shift_2", "lachman_1", "lachman_2"]

for name in dfs:
    import pandas as pd
    df = pd.read_csv(f"./analysis/{name}.csv", encoding = "utf-8")
    globals()[name] = df

In [161]:
pivot_shift_1.info()

<class 'pandas.DataFrame'>
RangeIndex: 38 entries, 0 to 37
Data columns (total 14 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  38 non-null     int64  
 1   id          38 non-null     int64  
 2   study       38 non-null     str    
 3   subgroup    38 non-null     str    
 4   ni          38 non-null     int64  
 5   0           38 non-null     float64
 6   1+          38 non-null     float64
 7   2+          38 non-null     int64  
 8   3+          38 non-null     int64  
 9   y           38 non-null     int64  
 10  xi          38 non-null     float64
 11  yi          38 non-null     float64
 12  se          38 non-null     float64
 13  sd          38 non-null     float64
dtypes: float64(6), int64(6), str(2)
memory usage: 5.1 KB


In [175]:
dfs = {
    "pivot_shift_1": pivot_shift_1, 
    "pivot_shift_2": pivot_shift_2, 
    "lachman_1": lachman_1,
    "lachman_2": lachman_2,
}

for name, df in dfs.items():
    df["group"] = df.groupby(["study"]).cumcount() + 1
    
    # reshape
    wide = (
        df.pivot(
            index= "study",
            columns="group",
            values=["subgroup", "ni", "yi", "sd"]
        )
    )
    
    # flatten column names
    wide.columns = [
        f"{name}{group}" if name != "subgroup" else f"treat{group}"
        for name, group in wide.columns
    ]
    
    # rename variables
    wide = (
        wide.rename(columns={
            "ni1": "n1",
            "yi1": "y1",
            "sd1": "sd1",
            "ni2": "n2",
            "yi2": "y2",
            "sd2": "sd2",
        })
        .reset_index()
    )

#    wide["n1"] = wide["n1"].astype(int)
#    wide["n2"] = wide["n2"].astype(int)
    wide["sd1"] = wide["sd1"].astype(float)
    wide["sd2"] =wide["sd2"].astype(float)
    wide["y1"] = wide["y1"].astype(float)
    wide["y2"] = wide["y2"].astype(float)
    
    import os
    os.makedirs(f"./analysis", exist_ok = True)
    wide.to_csv(f"./analysis/{name}_wide.csv", encoding = "utf-8")
    globals()[name] = wide

KeyError: "None of [Index(['subgroup', 'ni', 'yi', 'sd'], dtype='str')] are in the [columns]"

In [174]:
pivot_shift_1.info()

<class 'pandas.DataFrame'>
RangeIndex: 19 entries, 0 to 18
Data columns (total 9 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   study   19 non-null     str    
 1   treat1  19 non-null     str    
 2   treat2  19 non-null     str    
 3   n1      19 non-null     object 
 4   n2      19 non-null     object 
 5   y1      19 non-null     float64
 6   y2      19 non-null     float64
 7   sd1     19 non-null     float64
 8   sd2     19 non-null     float64
dtypes: float64(4), object(2), str(3)
memory usage: 1.9+ KB


In [176]:
import numpy as np

# Standardized Mean Difference (SMD)
def SMD(n1, n2, y1, y2, sd1, sd2):
    import numpy as np 
    sd = (
        (((n1 - 1) * sd1**2) + ((n2 - 1) * sd2**2))
        / (n1 + n2 - 2)
    )
    sp = np.sqrt(sd)
    return (np.abs(y1 - y2) / sp).round(3)


# Standard Error of the Standardized Mean Difference (SMD)

def seSMD(n1, n2, smd):
    se = np.sqrt(
        (n1 + n2)/(n1*n2)
        + smd**2/(2*(n1+n2-2))
    )

    se = round(se, 3)
    return se

dfs = {
    "pivot_shift_1": pivot_shift_1, 
    "pivot_shift_2": pivot_shift_2, 
    "lachman_1": lachman_1,
    "lachman_2": lachman_2,
}



for name, df in dfs.items():
    df["n1"] = df["n1"].astype(int)
    df["n2"] = df["n2"].astype(int)
    df["smd"] =  SMD(df["n1"], df["n2"],df["y1"], df["y2"],df["sd1"], df["sd2"])
    df["se_smd"] = seSMD(df["n1"], df["n2"], df["smd"])
    df.to_csv(f"./analysis/{name}_analysis.csv", encoding = "utf-8")

#### Graft failure

In [198]:
import pandas as pd

df = pd.read_csv(f"./data/graft_failure.csv", encoding = "utf-8")

In [199]:
df.head()

,id,study,subgroup,ni,xi,sdi
0,6,"Anz et al., 2023",BPTB,16,0,0.13
1,6,"Anz et al., 2023",HT,16,0,0.13
2,43,"Kautzner et al., 2015",BPTB,74,0,0.06
3,43,"Kautzner et al., 2015",HT,73,2,0.30
4,49,"Laxdal et al., 2005",BPTB,40,1,0.24


In [200]:
df["yi"] = round(((df["xi"] + 0.5)/(df["ni"] + 0.5)), 3)

In [201]:
def SE(p, n):
    import numpy as np
    se = (p * (1 - p)) / n
    se = np.sqrt(se)
    se = round(se, 3)
    return se

def SD(se, n):
    import numpy as np
    sd = se * np.sqrt(n)
    sd = round(sd, 3)
    return sd

In [202]:
p = df["yi"]
n = df["ni"]

se = SE(p, n)
sd = SD(se, n)
df["sdi"] = sd

In [203]:
df.head()

,id,study,subgroup,ni,xi,sdi,yi
0,6,"Anz et al., 2023",BPTB,16,0,0.172,0.030
1,6,"Anz et al., 2023",HT,16,0,0.172,0.030
2,43,"Kautzner et al., 2015",BPTB,74,0,0.086,0.007
3,43,"Kautzner et al., 2015",HT,73,2,0.179,0.034
4,49,"Laxdal et al., 2005",BPTB,40,1,0.190,0.037


In [204]:
graft_failure = df

In [205]:
dfs = {
    "graft_failure": graft_failure
}

for name, df in dfs.items():
    df["group"] = df.groupby(["study"]).cumcount() + 1
    
    # reshape
    wide = (
        df.pivot(
            index= "study",
            columns="group",
            values=["subgroup", "ni", "yi", "sdi"]
        )
    )
    
    # flatten column names
    wide.columns = [
        f"{name}{group}" if name != "subgroup" else f"treat{group}"
        for name, group in wide.columns
    ]
    
    # rename variables
    wide = (
        wide.rename(columns={
            "ni1": "n1",
            "yi1": "y1",
            "sdi1": "sd1",
            "ni2": "n2",
            "yi2": "y2",
            "sdi2": "sd2",
        })
        .reset_index()
    )

    wide["n1"] = wide["n1"].astype(int)
    wide["n2"] = wide["n2"].astype(int)
    wide["sd1"] = wide["sd1"].astype(float)
    wide["sd2"] =wide["sd2"].astype(float)
    wide["y1"] = wide["y1"].astype(float)
    wide["y2"] = wide["y2"].astype(float)
    
    import os
    os.makedirs(f"./analysis", exist_ok = True)
    wide.to_csv(f"./analysis/{name}_wide.csv", encoding = "utf-8")
    globals()[name] = wide

In [206]:
graft_failure.head()

,study,treat1,treat2,n1,n2,y1,y2,sd1,sd2
0,"Anz et al., 2023",BPTB,HT,16,16,0.030,0.030,0.172,0.172
1,"Kautzner et al., 2015",BPTB,HT,74,73,0.007,0.034,0.086,0.179
2,"Laxdal et al., 2005",BPTB,HT,40,78,0.037,0.032,0.190,0.177
3,"Lucidi et al., 2025",BPTB,HT,19,20,0.385,0.171,0.488,0.376
4,"Mohtadi et al., 2016",BPTB,HT,110,110,0.176,0.222,0.378,0.420


In [207]:
import numpy as np

# Standardized Mean Difference (SMD)
def SMD(n1, n2, y1, y2, sd1, sd2):
    import numpy as np 
    sd = (
        (((n1 - 1) * sd1**2) + ((n2 - 1) * sd2**2))
        / (n1 + n2 - 2)
    )
    sp = np.sqrt(sd)
    return (np.abs(y1 - y2) / sp).round(3)


# Standard Error of the Standardized Mean Difference (SMD)

def seSMD(n1, n2, smd):
    se = np.sqrt(
        (n1 + n2)/(n1*n2)
        + smd**2/(2*(n1+n2-2))
    )

    se = round(se, 3)
    return se

dfs = {
    "graft_failure": graft_failure
}

for name, df in dfs.items():
    df["smd"] =  SMD(df["n1"], df["n2"],df["y1"], df["y2"],df["sd1"], df["sd2"])
    df["se_smd"] = seSMD(df["n1"], df["n2"], df["smd"])
    df.to_csv(f"./analysis/{name}_analysis.csv", encoding = "utf-8")